In [ ]:
import numpy as np

def find_params(matrix_PSD_total, matrix_FC_total,
                array_MCI_PSD, array_HC_PSD,
                array_MCI_FC, array_HC_FC,
                normstrat = 'standard', weight=1.0):
    """
    Find model parameters that best reproduce empirical PSD and FC data
    for MCI and HC groups via inversion based on minimal squared difference.
    """
    
    # Normalize matrices globally
    if normstrat = 'standard'
        normalized_matrix_PSD = (matrix_PSD_total - matrix_PSD_total.mean()) / matrix_PSD_total.std() 
        normalized_matrix_FC  = (matrix_FC_total  - matrix_FC_total.min())  / matrix_FC_total.std()
    
    elif normstrat = 'minmax'    
        normalized_matrix_PSD = (matrix_PSD_total - matrix_PSD_total.min()) / (matrix_PSD_total.max() - matrix_PSD_total.min())
        normalized_matrix_FC  = (matrix_FC_total  - matrix_FC_total.min())  / (matrix_FC_total.max()  - matrix_FC_total.min())

    # Normalize empirical data (z-scored across all subjects)
    array_FC = np.concatenate((array_HC_FC, array_MCI_FC))
    array_PSD = np.concatenate((array_HC_PSD, array_MCI_PSD))
    
    normalized_array_HC_FC  = (array_HC_FC  - array_FC.mean()) / array_FC.std()
    normalized_array_MCI_FC = (array_MCI_FC - array_FC.mean()) / array_FC.std()
    normalized_array_HC_PSD  = (array_HC_PSD  - array_PSD.mean()) / array_PSD.std()
    normalized_array_MCI_PSD = (array_MCI_PSD - array_PSD.mean()) / array_PSD.std()

    # Initialize output containers
    closest_indices_HC, closest_indices_MCI = [], []

    # --- MCI inversion
    for index in range(len(normalized_array_MCI_PSD)):
        # Joint difference (summed over spatial dims)
        absolute_diff = ((normalized_matrix_PSD - normalized_array_MCI_PSD[index])**2 +
                         weight * (normalized_matrix_FC - normalized_array_MCI_FC[index])**2)
        absolute_diff_sum = absolute_diff.sum(axis=(0,1))
        min_idx = np.argmin(absolute_diff_sum)
        min_indices = np.unravel_index(min_idx, (normalized_matrix_PSD.shape[2],))
        closest_indices_MCI.append(min_indices)

    # --- HC inversion
    for index in range(len(normalized_array_HC_PSD)):
        absolute_diff = ((normalized_matrix_PSD - normalized_array_HC_PSD[index])**2 +
                         weight * (normalized_matrix_FC - normalized_array_HC_FC[index])**2)
        absolute_diff_sum = absolute_diff.sum(axis=(0,1))
        min_idx = np.argmin(absolute_diff_sum)
        min_indices = np.unravel_index(min_idx, (normalized_matrix_PSD.shape[2],))
        closest_indices_HC.append(min_indices)

    closest_indices_MCI = np.array(closest_indices_MCI)
    closest_indices_HC  = np.array(closest_indices_HC)

    return closest_indices_HC, closest_indices_MCI
